In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schemas/volumes are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

SILVER_TBL = f"{CATALOG}.`02_silver`.trip_updates"
GOLD_TBL = f"{CATALOG}.`03_gold`.trip_updates"
SILVER_STOPS_TBL = f"{CATALOG}.`02_silver`.stops"
CHECKPOINT = f"/Volumes/{CATALOG}/03_gold/_checkpoints/trip_updates"

PAYLOAD = ["arrival_time","arrival_delay_sec","departure_time","departure_delay_sec","stop_rel"]

# Pre-create the schema and volume if they are not there
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.`03_gold`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`03_gold`._checkpoints")

# The next incremental operation requires the table pre-exists, so we create an empty table if it's not there
spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD_TBL} AS SELECT * FROM {SILVER_TBL} WHERE 1=0")

def upsert_changes(batch_df, batch_id):
    keys = batch_df.select("entity_id","stop_sequence").distinct()
    seed = (spark.table(GOLD_TBL).join(keys, ["entity_id","stop_sequence"])  # only touched keys
              .withColumn("rn", F.row_number().over(
                  Window.partitionBy("entity_id","stop_sequence").orderBy(F.col("feed_ts").desc())))
              .filter("rn = 1").drop("rn"))

    w = Window.partitionBy("entity_id","stop_sequence").orderBy("feed_ts")

    changes = (seed.unionByName(batch_df, allowMissingColumns=True)
        .withColumn("prev", F.lag(F.struct(*PAYLOAD)).over(w))
        .filter(f"prev IS NULL OR struct({','.join(PAYLOAD)}) IS DISTINCT FROM prev"))

    changes = changes.drop("prev")

    (DeltaTable.forName(spark, GOLD_TBL).alias("g")
        .merge(changes.alias("n"),
               "g.entity_id=n.entity_id AND g.stop_sequence=n.stop_sequence AND g.feed_ts=n.feed_ts")
        .whenNotMatchedInsertAll().execute())

q = (spark.readStream.option("skipChangeCommits","true")
    .table(SILVER_TBL)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .foreachBatch(upsert_changes).trigger(availableNow=True)
    .start()
)

q.awaitTermination()


```sql

query = f"""
CREATE OR REPLACE TABLE {GOLD_TBL} AS
WITH ordered AS (
  SELECT *,
    LAG(struct(arrival_time, arrival_delay_sec, departure_time,
               departure_delay_sec, stop_rel))
      OVER (PARTITION BY entity_id, stop_sequence ORDER BY feed_ts) AS prev
  FROM {SILVER_TBL}
),
updates AS (
    SELECT * EXCEPT(prev) FROM ordered
WHERE prev IS NULL
   OR struct(arrival_time, arrival_delay_sec, departure_time,
    departure_delay_sec, stop_rel) IS DISTINCT
    FROM prev
)
SELECT
    u.feed_ts,
    entity_id,
    trip_id,
    route_id,
    start_date,
    start_time,               -- keep as string: GTFS allows >24:00:00
    trip_rel,
    stop_sequence,                  -- sparse / non-contiguous within a trip
    stop_id,
    s.stop_name,
    s.stop_lat,
    s.stop_lon,
    arrival_time,
    arrival_delay_sec,
    departure_time,
    departure_delay_sec,
    stop_rel,
    event_time,            -- coalesce(arrival, departure) for sorting/clustering
    source_file,
    _ingest_ts,
    _silver_ts,
    current_timestamp() AS _gold_ts
FROM updates u
LEFT JOIN {SILVER_STOPS_TBL} s
ON u.stop_id = s.stop_id
"""


spartk.sql(query)

```